# 17 — Distribution-shift analysis

This notebook diagnoses why development regimes—especially run `baa2a6f4-7121-4bce-9c80-611ebc22ceaf`—differ from training. It does not change data, labels, features, splits, preprocessing, models, or thresholds, and it never evaluates test predictions.

### 1. Define protected paths and analysis configuration

**What this cell does:** Imports diagnostic libraries, resolves the current repository paths, records checksums for all protected inputs, and centralizes the difficult-run ID and Rule C thresholds.  
**Why it matters:** Distribution analysis must use the current 153k experiment and the exact historical label definition without silently changing the pipeline.  
**What to understand:** The test files are fingerprinted only for immutability; no test labels, probabilities, or model performance are loaded.

In [1]:
from pathlib import Path
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from IPython.display import display

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "interim" / "feature_dataset.csv").exists() and (PROJECT_ROOT.parent / "data" / "interim" / "feature_dataset.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data" / "interim" / "feature_dataset.csv").exists():
    raise FileNotFoundError("Could not locate the AdoptAI_V1 repository root.")

PATHS = {
    "feature_dataset": PROJECT_ROOT / "data/interim/feature_dataset.csv",
    "train": PROJECT_ROOT / "data/modeling/train.csv",
    "validation": PROJECT_ROOT / "data/modeling/validation.csv",
    "train_identifiers": PROJECT_ROOT / "data/modeling/preprocessed/train_identifiers.csv",
    "validation_identifiers": PROJECT_ROOT / "data/modeling/preprocessed/validation_identifiers.csv",
    "x_train": PROJECT_ROOT / "data/modeling/preprocessed/X_train_tree.csv",
    "x_validation": PROJECT_ROOT / "data/modeling/preprocessed/X_validation_tree.csv",
    "baseline_model": PROJECT_ROOT / "models/baseline/lightgbm.joblib",
    "validation_predictions": PROJECT_ROOT / "reports/baseline_validation_predictions.csv",
    "all_runs": PROJECT_ROOT / "reports/all_runs_split_diagnostic.csv",
    "robustness_by_run": PROJECT_ROOT / "reports/robustness_by_run.csv",
    "robustness_by_machine": PROJECT_ROOT / "reports/robustness_by_machine.csv",
    "preprocessing_features": PROJECT_ROOT / "reports/preprocessing_feature_report.csv",
    "segmented": PROJECT_ROOT / "data/interim/segmented_metrics.csv",
    "run_summary": PROJECT_ROOT / "reports/run_summary.csv",
}
TEST_PATHS = [
    PROJECT_ROOT / "data/modeling/test.csv",
    PROJECT_ROOT / "data/modeling/preprocessed/X_test_tree.csv",
    PROJECT_ROOT / "data/modeling/preprocessed/X_test_linear.csv",
    PROJECT_ROOT / "data/modeling/preprocessed/y_test.csv",
    PROJECT_ROOT / "data/modeling/preprocessed/test_identifiers.csv",
]
REPORT_DIR = PROJECT_ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures/distribution_shift"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
DIFFICULT_RUN = "baa2a6f4-7121-4bce-9c80-611ebc22ceaf"
TARGET = "slowdown_in_5min"

RULE_C_CONFIG = {
    "moderate_window_seconds": 30, "severe_window_seconds": 60, "minimum_observations": 3,
    "moderate": {"cpu_pct_mean": 80.0, "ram_pct_mean": 85.0, "swap_pct_mean": 60.0,
                 "disk_latency_ms_mean": 5.0, "context_run_quantile": 0.95, "context_median_multiplier": 1.5},
    "severe": {"cpu_pct_mean": 95.0, "ram_pct_mean": 95.0, "swap_pct_mean": 70.0,
               "disk_latency_ms_mean": 20.0, "context_run_quantile": 0.99, "context_median_multiplier": 2.0},
}

def sha256(path, block=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(block), b""):
            digest.update(chunk)
    return digest.hexdigest()

missing = [str(path) for path in list(PATHS.values()) + TEST_PATHS if not path.exists()]
assert not missing, f"Required artifacts are missing: {missing}"
input_checksums_before = {name: sha256(path) for name, path in PATHS.items()}
test_checksums_before = {str(path.relative_to(PROJECT_ROOT)): sha256(path) for path in TEST_PATHS}
print(f"Repository: {PROJECT_ROOT}")
print("Exact Rule C configuration:")
display(pd.json_normalize(RULE_C_CONFIG, sep=".").T.rename(columns={0: "value"}))
print("Protected test artifacts fingerprinted, but their rows, labels, probabilities, and performance are not loaded.")

Repository: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1
Exact Rule C configuration:


,value
moderate_window_seconds,30.00
severe_window_seconds,60.00
minimum_observations,3.00
moderate.cpu_pct_mean,80.00
moderate.ram_pct_mean,85.00
moderate.swap_pct_mean,60.00
moderate.disk_latency_ms_mean,5.00
moderate.context_run_quantile,0.95
moderate.context_median_multiplier,1.50
severe.cpu_pct_mean,95.00


Protected test artifacts fingerprinted, but their rows, labels, probabilities, and performance are not loaded.


### 2. Load and validate only current development artifacts

**What this cell does:** Loads train, validation, identifiers, preprocessing metadata, existing validation predictions, and diagnostic reports; it checks their row alignment and current-experiment sizes.  
**Why it matters:** Mixing the old 82k artifacts with the current experiment would invalidate every comparison.  
**What to understand:** The 22,212-row validation prediction file aligns one-to-one with current validation; a 6,640-row historical artifact would fail immediately.

In [2]:
feature_header = pd.read_csv(PATHS["feature_dataset"], nrows=0)
feature_row_count = sum(1 for _ in open(PATHS["feature_dataset"], encoding="utf-8")) - 1
train = pd.read_csv(PATHS["train"])
validation = pd.read_csv(PATHS["validation"])
train_ids = pd.read_csv(PATHS["train_identifiers"])
validation_ids = pd.read_csv(PATHS["validation_identifiers"])
predictions = pd.read_csv(PATHS["validation_predictions"])
all_runs = pd.read_csv(PATHS["all_runs"])
robustness_run = pd.read_csv(PATHS["robustness_by_run"])
robustness_machine = pd.read_csv(PATHS["robustness_by_machine"])
preprocessing_features = pd.read_csv(PATHS["preprocessing_features"])

assert feature_row_count == 133_016, f"Expected current 133,016-row feature dataset, found {feature_row_count:,}."
assert len(train) == len(train_ids) == 82_860
assert len(validation) == len(validation_ids) == len(predictions) == 22_212
assert len(predictions) != 6_640, "Historical 82k validation predictions detected."
assert PATHS["baseline_model"].stat().st_size > 0
required = {"machine_id", "run_id", "segment_id", "timestamp", TARGET}
assert required.issubset(train.columns) and required.issubset(validation.columns)
assert predictions[["machine_id", "run_id", "segment_id", "timestamp"]].astype(str).equals(
    validation[["machine_id", "run_id", "segment_id", "timestamp"]].astype(str).reset_index(drop=True)
)
assert np.array_equal(predictions["true_target"].to_numpy(), validation[TARGET].astype(int).to_numpy())

for frame in (train, validation):
    frame["_timestamp_dt"] = pd.to_datetime(frame["timestamp"], errors="coerce", utc=True)
    assert frame["_timestamp_dt"].notna().all()
development = pd.concat([train.assign(analysis_split="train"), validation.assign(analysis_split="validation")], ignore_index=True)
difficult = validation.loc[validation["run_id"].eq(DIFFICULT_RUN)].copy()
assert len(difficult) == 8_602

groups = {
    "train_overall": train,
    "train_positive": train.loc[train[TARGET].eq(1)],
    "train_negative": train.loc[train[TARGET].eq(0)],
    "difficult_overall": difficult,
    "difficult_positive": difficult.loc[difficult[TARGET].eq(1)],
    "difficult_negative": difficult.loc[difficult[TARGET].eq(0)],
}
low_prevalence_validation_runs = (
    validation.groupby(["machine_id", "run_id"], as_index=False)[TARGET]
    .agg(rows="size", positive_rate="mean")
    .query("positive_rate <= 0.15")
    .sort_values(["machine_id", "positive_rate"])
)
artifact_audit = pd.DataFrame([
    {"artifact": key, "path": str(path.relative_to(PROJECT_ROOT)), "exists": path.exists()}
    for key, path in PATHS.items()
])
print(f"Current feature dataset: {feature_row_count:,} rows and {len(feature_header.columns)} columns")
print(f"Development analysis: {len(development):,} rows; difficult run: {len(difficult):,} rows")
display(pd.DataFrame({name: {"rows": len(frame), "positive_rate": frame[TARGET].mean()} for name, frame in groups.items()}).T)
print("Low-prevalence validation regimes:")
display(low_prevalence_validation_runs)

Current feature dataset: 133,016 rows and 251 columns
Development analysis: 105,072 rows; difficult run: 8,602 rows


,rows,positive_rate
train_overall,82860.0,0.340309
train_positive,28198.0,1.000000
train_negative,54662.0,0.000000
difficult_overall,8602.0,0.203092
difficult_positive,1747.0,1.000000
difficult_negative,6855.0,0.000000


Low-prevalence validation regimes:


,machine_id,run_id,rows,positive_rate
0,0890dcc046c079acc4de4202,90c048bb-46c5-4345-a113-d3bc37b876cb,3024,0.0


### 3. Quantify raw and engineered feature shifts

**What this cell does:** Compares train with the difficult run, then repeats the comparison conditionally for positives and negatives using standardized mean difference, PSI, and KS distance.  
**Why it matters:** Overall shift indicates changed operating conditions, while target-conditional shift can reveal that slowdown and non-slowdown examples have changed differently.  
**What to understand:** Large samples make p-values unhelpful, so ranking uses effect-size and distribution-distance measures plus explicit missingness differences.

In [3]:
RAW_FEATURES = [c for c in [
    "cpu_pct", "cpu_frequency_mhz", "ram_pct", "ram_used_mb", "ram_available_mb",
    "swap_pct", "swap_used_mb", "disk_usage_pct", "disk_free_gb", "disk_read_mb_s",
    "disk_write_mb_s", "disk_latency_ms", "net_sent_mb_s", "net_recv_mb_s",
    "network_latency_ms", "process_count", "thread_count", "context_switches_per_s",
    "battery_pct", "battery_plugged"
] if c in train.columns]
retained = preprocessing_features.loc[preprocessing_features["retained_or_removed"].eq("retained"), "feature_name"].tolist()
MODEL_FEATURES = [c for c in retained if c in train.columns and pd.api.types.is_numeric_dtype(train[c])]
assert len(MODEL_FEATURES) == 228

def numeric_values(frame, feature):
    return pd.to_numeric(frame[feature], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna().to_numpy(dtype=float)

def psi(reference, comparison, bins=10):
    if len(reference) < 2 or len(comparison) < 2:
        return np.nan
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0 if np.allclose(np.nanmedian(reference), np.nanmedian(comparison)) else np.nan
    edges[0], edges[-1] = -np.inf, np.inf
    ref = np.histogram(reference, bins=edges)[0] / len(reference)
    cmp = np.histogram(comparison, bins=edges)[0] / len(comparison)
    ref, cmp = np.clip(ref, 1e-6, None), np.clip(cmp, 1e-6, None)
    return float(np.sum((cmp - ref) * np.log(cmp / ref)))

def describe_comparison(feature, left, right, comparison_name):
    a, b = numeric_values(left, feature), numeric_values(right, feature)
    pooled = np.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1)) / 2) if len(a) > 1 and len(b) > 1 else np.nan
    smd = (np.mean(b) - np.mean(a)) / pooled if np.isfinite(pooled) and pooled > 0 else 0.0
    ks = float(ks_2samp(a, b).statistic) if len(a) and len(b) else np.nan
    psi_value = psi(a, b)
    score = abs(smd) + min(psi_value, 5.0) + ks if np.isfinite(psi_value) and np.isfinite(ks) else abs(smd)
    def q(x, p): return float(np.quantile(x, p)) if len(x) else np.nan
    return {
        "comparison": comparison_name, "feature": feature,
        "feature_type": "raw" if feature in RAW_FEATURES else "engineered",
        "train_mean": np.mean(a) if len(a) else np.nan, "difficult_mean": np.mean(b) if len(b) else np.nan,
        "train_median": np.median(a) if len(a) else np.nan, "difficult_median": np.median(b) if len(b) else np.nan,
        "train_std": np.std(a, ddof=1) if len(a) > 1 else np.nan, "difficult_std": np.std(b, ddof=1) if len(b) > 1 else np.nan,
        "train_p10": q(a,.10), "train_p25": q(a,.25), "train_p75": q(a,.75), "train_p90": q(a,.90), "train_p95": q(a,.95),
        "difficult_p10": q(b,.10), "difficult_p25": q(b,.25), "difficult_p75": q(b,.75), "difficult_p90": q(b,.90), "difficult_p95": q(b,.95),
        "train_missing_pct": float(left[feature].isna().mean()*100), "difficult_missing_pct": float(right[feature].isna().mean()*100),
        "missing_pct_difference": float((right[feature].isna().mean()-left[feature].isna().mean())*100),
        "standardized_mean_difference": float(smd), "absolute_smd": float(abs(smd)), "psi": psi_value,
        "ks_statistic": ks, "shift_metric": float(score),
    }

comparisons = [
    ("train_vs_difficult", groups["train_overall"], groups["difficult_overall"]),
    ("train_positive_vs_difficult_positive", groups["train_positive"], groups["difficult_positive"]),
    ("train_negative_vs_difficult_negative", groups["train_negative"], groups["difficult_negative"]),
]
ranking_rows = [describe_comparison(feature, left, right, name) for name,left,right in comparisons for feature in MODEL_FEATURES]
feature_ranking = pd.DataFrame(ranking_rows)
feature_ranking["shift_rank"] = feature_ranking.groupby("comparison")["shift_metric"].rank(method="first", ascending=False).astype(int)
feature_ranking = feature_ranking.sort_values(["comparison", "shift_rank"])
feature_ranking.to_csv(REPORT_DIR / "distribution_shift_feature_ranking.csv", index=False)

raw_shift = feature_ranking.loc[feature_ranking["feature_type"].eq("raw")].copy()
print("Top 20 overall shifted model features:")
display(feature_ranking.query("comparison == 'train_vs_difficult'").head(20)[
    ["shift_rank","feature","feature_type","train_median","difficult_median","absolute_smd","psi","ks_statistic","missing_pct_difference"]
])
print("Top raw shifts for each target-conditional comparison:")
display(raw_shift.groupby("comparison", group_keys=False).head(10)[["comparison","feature","shift_rank","shift_metric","absolute_smd","psi","ks_statistic"]])

Top 20 overall shifted model features:


,shift_rank,feature,feature_type,train_median,difficult_median,absolute_smd,psi,ks_statistic,missing_pct_difference
8,1,disk_free_gb,raw,186.840000,140.160000,1.772381,12.437531,1.000000,0.0
7,2,disk_usage_pct,raw,60.700000,70.600000,1.688567,12.528830,1.000000,0.0
218,3,thread_count_min_120s,engineered,3533.000000,4230.500000,1.350638,9.213622,0.712772,0.0
216,4,thread_count_mean_120s,engineered,3588.958333,4292.059195,1.345652,9.335114,0.714888,0.0
213,5,thread_count_min_60s,engineered,3546.000000,4249.000000,1.347473,8.492784,0.712484,0.0
208,6,thread_count_min_30s,engineered,3556.000000,4265.000000,1.345656,8.568112,0.710490,0.0
211,7,thread_count_mean_60s,engineered,3584.145614,4291.298148,1.343752,9.365410,0.709933,0.0
206,8,thread_count_mean_30s,engineered,3581.933333,4290.688095,1.342468,8.654209,0.707771,0.0
16,9,thread_count,raw,3580.000000,4288.000000,1.341229,8.573928,0.706367,0.0
207,10,thread_count_max_30s,engineered,3606.500000,4321.000000,1.337213,8.717340,0.706763,0.0


Top raw shifts for each target-conditional comparison:


,comparison,feature,shift_rank,shift_metric,absolute_smd,psi,ks_statistic
464,train_negative_vs_difficult_negative,disk_free_gb,1,8.030627,2.030627,12.444539,1.000000
463,train_negative_vs_difficult_negative,disk_usage_pct,2,7.923878,1.923878,11.683763,1.000000
472,train_negative_vs_difficult_negative,thread_count,9,7.390264,1.612960,9.685546,0.777304
459,train_negative_vs_difficult_negative,ram_used_mb,13,7.243008,1.607480,7.527811,0.635528
462,train_negative_vs_difficult_negative,swap_used_mb,14,6.978212,1.185082,9.298047,0.793130
457,train_negative_vs_difficult_negative,cpu_frequency_mhz,15,6.896434,1.325306,7.762516,0.571128
460,train_negative_vs_difficult_negative,ram_available_mb,25,6.734431,1.201860,5.362377,0.532571
458,train_negative_vs_difficult_negative,ram_pct,27,6.726084,1.197292,5.329931,0.528792
456,train_negative_vs_difficult_negative,cpu_pct,33,6.535915,0.977520,5.612667,0.558395
461,train_negative_vs_difficult_negative,swap_pct,44,5.944405,0.467510,10.645947,0.476895


### 4. Reproduce Rule C pressure components without changing Rule C

**What this cell does:** Recomputes the exact 30-second moderate and 60-second severe rolling pressure flags inside each machine/run/segment, then profiles their frequency for train and difficult-run target-positive rows.  
**Why it matters:** It reveals whether the difficult run reaches the unchanged future slowdown label through different resource-pressure combinations.  
**What to understand:** `missed_deadline`, sensor errors, temperature, and gap diagnostics are not pressure signals and are not used here.

In [4]:
rule_data = development[["machine_id","run_id","segment_id","timestamp",TARGET,
                         "cpu_pct","ram_pct","swap_pct","disk_latency_ms","context_switches_per_s"]].copy()
rule_data["_timestamp_dt"] = pd.to_datetime(rule_data["timestamp"], utc=True)
rule_data["_source_order"] = np.arange(len(rule_data))
keys = ["machine_id","run_id","segment_id"]
rule_data = rule_data.sort_values(keys + ["_timestamp_dt","_source_order"], kind="mergesort")
metrics = ["cpu_pct","ram_pct","swap_pct","disk_latency_ms","context_switches_per_s"]
for label, seconds in [("moderate",30),("severe",60)]:
    rolled_parts = []
    for _, segment in rule_data.groupby(keys, sort=False):
        ordered = segment.sort_values("_timestamp_dt")
        values = ordered.set_index("_timestamp_dt")[metrics].rolling(f"{seconds}s", min_periods=3).mean()
        values.index = ordered.index
        values.columns = [f"{c}_{label}_mean" for c in metrics]
        rolled_parts.append(values)
    rolled = pd.concat(rolled_parts).sort_index()
    rule_data.loc[rolled.index, rolled.columns] = rolled

context_base = rule_data.groupby(["machine_id","run_id"])["context_switches_per_s"].agg(
    context_run_median="median", context_run_p95=lambda s:s.quantile(.95), context_run_p99=lambda s:s.quantile(.99)
).reset_index()
rule_data = rule_data.merge(context_base, on=["machine_id","run_id"], how="left", validate="many_to_one")
mod, sev = RULE_C_CONFIG["moderate"], RULE_C_CONFIG["severe"]
rule_data["moderate_cpu"] = rule_data["cpu_pct_moderate_mean"].ge(mod["cpu_pct_mean"])
rule_data["moderate_ram"] = rule_data["ram_pct_moderate_mean"].ge(mod["ram_pct_mean"])
rule_data["moderate_swap"] = rule_data["swap_pct_moderate_mean"].ge(mod["swap_pct_mean"])
rule_data["moderate_disk"] = rule_data["disk_latency_ms_moderate_mean"].ge(mod["disk_latency_ms_mean"])
rule_data["moderate_context"] = rule_data["context_switches_per_s_moderate_mean"].ge(rule_data["context_run_p95"]) & rule_data["context_switches_per_s_moderate_mean"].ge(1.5*rule_data["context_run_median"])
rule_data["severe_cpu"] = rule_data["cpu_pct_severe_mean"].ge(sev["cpu_pct_mean"])
rule_data["severe_ram"] = rule_data["ram_pct_severe_mean"].ge(sev["ram_pct_mean"])
rule_data["severe_swap"] = rule_data["swap_pct_severe_mean"].ge(sev["swap_pct_mean"])
rule_data["severe_disk"] = rule_data["disk_latency_ms_severe_mean"].ge(sev["disk_latency_ms_mean"])
rule_data["severe_context"] = rule_data["context_switches_per_s_severe_mean"].ge(rule_data["context_run_p99"]) & rule_data["context_switches_per_s_severe_mean"].ge(2.0*rule_data["context_run_median"])
moderate_cols = [f"moderate_{x}" for x in ["cpu","ram","swap","disk","context"]]
severe_cols = [f"severe_{x}" for x in ["cpu","ram","swap","disk","context"]]
rule_data["moderate_signal_count"] = rule_data[moderate_cols].sum(axis=1)
rule_data["severe_signal_count"] = rule_data[severe_cols].sum(axis=1)
rule_data["rule_c_now_recreated"] = rule_data["severe_signal_count"].ge(1) | rule_data["moderate_signal_count"].ge(2)

profile_groups = {
    "train_positive_future_label": rule_data[rule_data["analysis_split"].eq("train") & rule_data[TARGET].eq(1)] if "analysis_split" in rule_data else None,
}
# analysis_split is recovered from run membership because the compact rolling frame intentionally excludes unrelated columns.
train_run_set = set(train["run_id"])
rule_data["analysis_split"] = np.where(rule_data["run_id"].isin(train_run_set), "train", "validation")
profile_groups = {
    "train_positive_future_label": rule_data[rule_data["analysis_split"].eq("train") & rule_data[TARGET].eq(1)],
    "difficult_positive_future_label": rule_data[rule_data["run_id"].eq(DIFFICULT_RUN) & rule_data[TARGET].eq(1)],
    "train_all": rule_data[rule_data["analysis_split"].eq("train")],
    "difficult_all": rule_data[rule_data["run_id"].eq(DIFFICULT_RUN)],
}
profile_rows=[]
for group_name, frame in profile_groups.items():
    for signal in moderate_cols + severe_cols + ["rule_c_now_recreated"]:
        profile_rows.append({"profile_type":"signal_frequency","group":group_name,"condition":signal,
                             "row_count":len(frame),"triggered_count":int(frame[signal].sum()),"triggered_percentage":float(frame[signal].mean()*100)})
    combinations = frame[moderate_cols + severe_cols].apply(lambda row:"+".join(row.index[row].str.replace("moderate_","M:").str.replace("severe_","S:")) or "none", axis=1)
    for combo,count in combinations.value_counts().head(10).items():
        profile_rows.append({"profile_type":"top_pressure_combination","group":group_name,"condition":combo,
                             "row_count":len(frame),"triggered_count":int(count),"triggered_percentage":float(count/len(frame)*100)})
rule_c_profile = pd.DataFrame(profile_rows)
rule_c_profile.to_csv(REPORT_DIR / "difficult_run_rule_c_profile.csv", index=False)
print("Rule C pressure frequencies among future-label positives:")
display(rule_c_profile.query("profile_type == 'signal_frequency' and group in ['train_positive_future_label','difficult_positive_future_label']"))

Rule C pressure frequencies among future-label positives:


,profile_type,group,condition,row_count,triggered_count,triggered_percentage
0,signal_frequency,train_positive_future_label,moderate_cpu,28198,12171,43.162636
1,signal_frequency,train_positive_future_label,moderate_ram,28198,15354,54.450670
2,signal_frequency,train_positive_future_label,moderate_swap,28198,2558,9.071565
3,signal_frequency,train_positive_future_label,moderate_disk,28198,99,0.351089
4,signal_frequency,train_positive_future_label,moderate_context,28198,1760,6.241577
5,signal_frequency,train_positive_future_label,severe_cpu,28198,5586,19.809916
6,signal_frequency,train_positive_future_label,severe_ram,28198,2358,8.362295
7,signal_frequency,train_positive_future_label,severe_swap,28198,2214,7.851621
8,signal_frequency,train_positive_future_label,severe_disk,28198,95,0.336903
9,signal_frequency,train_positive_future_label,severe_context,28198,171,0.606426


### 5. Diagnose model probabilities and false negatives

**What this cell does:** Uses the already-created baseline validation probabilities, summarizes difficult-run true negatives/positives, and ranks features separating correctly detected positives from false negatives at 0.50.  
**Why it matters:** This distinguishes borderline misses from confidently missed slowdown regimes without optimizing a threshold.  
**What to understand:** The threshold is diagnostic only, and the existing model is not retrained.

In [5]:
dp = predictions.loc[predictions["run_id"].eq(DIFFICULT_RUN)].copy()
assert len(dp) == len(difficult)
prob_col = "lightgbm_predicted_probability"
dp["diagnostic_outcome"] = np.select(
    [(dp.true_target.eq(1)&dp[prob_col].ge(.5)), (dp.true_target.eq(1)&dp[prob_col].lt(.5)),
     (dp.true_target.eq(0)&dp[prob_col].ge(.5)), (dp.true_target.eq(0)&dp[prob_col].lt(.5))],
    ["true_positive","false_negative","false_positive","true_negative"], default="unclassified"
)
prob_rows=[]
for label, frame in [("true_negative",dp[dp.true_target.eq(0)]),("true_positive_actual",dp[dp.true_target.eq(1)]),
                     ("correctly_detected_positive",dp[dp.diagnostic_outcome.eq("true_positive")]),
                     ("false_negative",dp[dp.diagnostic_outcome.eq("false_negative")])]:
    values=frame[prob_col]
    row={"group":label,"rows":len(frame),"mean_probability":values.mean(),"median_probability":values.median(),
         "p10":values.quantile(.10),"p25":values.quantile(.25),"p75":values.quantile(.75),"p90":values.quantile(.90)}
    for threshold in [.2,.3,.4,.5]: row[f"count_probability_below_{str(threshold).replace('.','_')}"]=int((values<threshold).sum())
    prob_rows.append(row)
probability_profile=pd.DataFrame(prob_rows)
probability_profile.to_csv(REPORT_DIR / "difficult_run_probability_profile.csv",index=False)

difficult_with_outcome = difficult.reset_index(drop=True).copy()
difficult_with_outcome["diagnostic_outcome"] = dp["diagnostic_outcome"].to_numpy()
tp_frame=difficult_with_outcome[difficult_with_outcome.diagnostic_outcome.eq("true_positive")]
fn_frame=difficult_with_outcome[difficult_with_outcome.diagnostic_outcome.eq("false_negative")]
fn_rows=[]
for feature in MODEL_FEATURES:
    result=describe_comparison(feature,tp_frame,fn_frame,"difficult_true_positive_vs_false_negative")
    result.update({"true_positive_rows":len(tp_frame),"false_negative_rows":len(fn_frame)})
    fn_rows.append(result)
false_negative_profile=pd.DataFrame(fn_rows).sort_values("shift_metric",ascending=False).head(15).copy()
false_negative_profile["shift_rank"]=np.arange(1,len(false_negative_profile)+1)
false_negative_profile.to_csv(REPORT_DIR / "difficult_run_false_negative_profile.csv",index=False)
display(probability_profile)
print("Top features distinguishing detected positives from false negatives:")
display(false_negative_profile[["shift_rank","feature","feature_type","train_median","difficult_median","absolute_smd","psi","ks_statistic"]])

,group,rows,mean_probability,median_probability,p10,p25,p75,p90,count_probability_below_0_2,count_probability_below_0_3,count_probability_below_0_4,count_probability_below_0_5
0,true_negative,6855,0.209334,0.142208,0.031955,0.064667,0.295020,0.482556,4269,5172,5762,6233
1,true_positive_actual,1747,0.505927,0.425698,0.078227,0.151710,0.910934,0.991106,526,689,852,947
2,correctly_detected_positive,800,0.863709,0.933451,0.612584,0.779358,0.987800,0.996744,0,0,0,0
3,false_negative,947,0.203682,0.170557,0.051620,0.096658,0.310773,0.400213,526,689,852,947


Top features distinguishing detected positives from false negatives:


,shift_rank,feature,feature_type,train_median,difficult_median,absolute_smd,psi,ks_statistic
20,1,cpu_pct_mean_30s,engineered,72.439583,52.966667,1.430165,3.888318,0.508153
46,2,ram_pct_max_120s,engineered,92.200000,85.100000,1.278055,3.824023,0.546325
42,3,ram_pct_min_60s,engineered,88.300000,82.200000,0.817341,4.093268,0.489967
30,4,cpu_pct_mean_120s,engineered,68.122576,53.913953,1.288270,3.401445,0.457042
22,5,cpu_pct_min_30s,engineered,59.000000,39.900000,1.209858,3.456528,0.442293
25,6,cpu_pct_mean_60s,engineered,70.862471,52.742308,1.420897,3.128344,0.499810
37,7,ram_pct_min_30s,engineered,89.150000,82.400000,0.974307,3.365448,0.530075
47,8,ram_pct_min_120s,engineered,86.400000,81.800000,0.586084,3.773024,0.444322
27,9,cpu_pct_min_60s,engineered,54.300000,37.800000,1.070913,3.259193,0.426707
45,10,ram_pct_mean_120s,engineered,89.778755,83.023729,0.922413,3.198471,0.504385


### 6. Summarize machine regimes and run drift over time

**What this cell does:** Aggregates development rows by machine and chronological run, including label prevalence, raw-feature levels, and run-to-run prevalence variability.  
**Why it matters:** A difficult run may be an isolated anomaly, a gradual drift, or a recurring machine-specific operating regime.  
**What to understand:** These tables use train and validation only; test rows and test metrics are excluded.

In [6]:
major_raw = [c for c in ["cpu_pct","ram_pct","swap_pct","disk_latency_ms","disk_read_mb_s","disk_write_mb_s","context_switches_per_s","process_count","thread_count"] if c in development]
run_base = development.groupby(["machine_id","run_id"],as_index=False).agg(
    start_timestamp=("_timestamp_dt","min"), end_timestamp=("_timestamp_dt","max"), row_count=(TARGET,"size"),
    positive_count=(TARGET,"sum"), positive_rate=(TARGET,"mean"), segment_count=("segment_id","nunique"), split=("analysis_split","first")
)
for feature in major_raw:
    stats=development.groupby(["machine_id","run_id"])[feature].agg(median="median",p90=lambda s:s.quantile(.90),p95=lambda s:s.quantile(.95)).reset_index()
    stats=stats.rename(columns={"median":f"{feature}_median","p90":f"{feature}_p90","p95":f"{feature}_p95"})
    run_base=run_base.merge(stats,on=["machine_id","run_id"],how="left")
run_base["chronological_position"] = run_base.sort_values("start_timestamp").groupby("machine_id").cumcount()+1
run_distribution_drift=run_base.sort_values(["machine_id","start_timestamp"])
run_distribution_drift.to_csv(REPORT_DIR / "run_distribution_drift.csv",index=False)

machine_rows=[]
for machine_id,frame in development.groupby("machine_id"):
    runs=run_distribution_drift[run_distribution_drift.machine_id.eq(machine_id)]
    row={"machine_id":machine_id,"development_rows":len(frame),"positive_rate":frame[TARGET].mean(),"run_count":frame.run_id.nunique(),
         "run_positive_rate_mean":runs.positive_rate.mean(),"run_positive_rate_std":runs.positive_rate.std(),
         "run_positive_rate_min":runs.positive_rate.min(),"run_positive_rate_max":runs.positive_rate.max()}
    raw_scores=[]
    for feature in major_raw:
        vals=numeric_values(frame,feature); ref=numeric_values(development,feature)
        row[f"{feature}_median"]=np.median(vals) if len(vals) else np.nan
        row[f"{feature}_p90"]=np.quantile(vals,.90) if len(vals) else np.nan
        row[f"{feature}_p95"]=np.quantile(vals,.95) if len(vals) else np.nan
        pooled=np.sqrt((np.var(ref,ddof=1)+np.var(vals,ddof=1))/2)
        raw_scores.append(abs((np.mean(vals)-np.mean(ref))/pooled) if pooled>0 else 0)
    row["median_absolute_raw_smd_vs_development"]=float(np.median(raw_scores))
    row["maximum_absolute_raw_smd_vs_development"]=float(np.max(raw_scores))
    machine_rows.append(row)
machine_summary=pd.DataFrame(machine_rows).sort_values("machine_id")
machine_summary.to_csv(REPORT_DIR / "machine_distribution_summary.csv",index=False)
display(machine_summary[["machine_id","development_rows","positive_rate","run_count","run_positive_rate_std","run_positive_rate_min","run_positive_rate_max","median_absolute_raw_smd_vs_development"]])

,machine_id,development_rows,positive_rate,run_count,run_positive_rate_std,run_positive_rate_min,run_positive_rate_max,median_absolute_raw_smd_vs_development
0,0890dcc046c079acc4de4202,25499,0.133652,7,0.461560,0.000000,1.0,0.575279
1,7232bc533c21ce408d45d473,13285,0.397516,11,0.281577,0.047619,1.0,0.798357
2,a0f8c86097e55fbfa506d057,50324,0.397941,13,0.396088,0.000000,1.0,0.128243
3,d588df123ac0d0ce20b112ac,15964,0.651215,15,0.482928,0.000000,1.0,0.259150


### 7. Check difficult-run data quality against neighboring development runs

**What this cell does:** Reads raw segmented measurements and measures timing, gaps, duplicates, missingness, constants, impossible values, and collector/sensor abnormalities for the difficult run and its nearest preceding development neighbors.  
**Why it matters:** Distribution shift should not be mistaken for corrupted collection.  
**What to understand:** The classification is evidence-based and diagnostic; no row is removed or repaired automatically.

In [7]:
segmented = pd.read_csv(PATHS["segmented"])
segmented["_timestamp_dt"] = pd.to_datetime(segmented["timestamp"],errors="coerce",utc=True)
difficult_meta=all_runs.loc[all_runs.run_id.eq(DIFFICULT_RUN)].iloc[0]
same_machine_dev=all_runs[(all_runs.machine_id.eq(difficult_meta.machine_id)) & (all_runs.current_split.ne("test"))].sort_values("chronological_position_within_machine")
position=int(difficult_meta.chronological_position_within_machine)
neighbor_ids=same_machine_dev[same_machine_dev.chronological_position_within_machine.lt(position)].tail(2).run_id.tolist()
quality_run_ids=neighbor_ids+[DIFFICULT_RUN]
quality_metrics=[c for c in RAW_FEATURES if c in segmented.columns]
quality_rows=[]
for run_id in quality_run_ids:
    frame=segmented[segmented.run_id.eq(run_id)].sort_values("_timestamp_dt").copy()
    delta=frame["_timestamp_dt"].diff().dt.total_seconds()
    positive_delta=delta[delta.gt(0)]
    median_interval=positive_delta.median()
    gap_threshold=max(5*median_interval,10) if pd.notna(median_interval) else np.nan
    impossible=0
    for feature in [c for c in ["cpu_pct","ram_pct","swap_pct","disk_usage_pct","battery_pct"] if c in frame]:
        impossible += int(((frame[feature]<0)|(frame[feature]>100)).sum())
    for feature in [c for c in ["disk_read_mb_s","disk_write_mb_s","disk_latency_ms","net_sent_mb_s","net_recv_mb_s","network_latency_ms","process_count","thread_count","context_switches_per_s"] if c in frame]:
        impossible += int((frame[feature]<0).sum())
    sensor_text=frame.get("sensor_errors_json",pd.Series(index=frame.index,dtype=object)).fillna("").astype(str).str.strip().str.lower()
    sensor_abnormal=~sensor_text.isin(["","{}","[]","null","nan","none"])
    sample_unreliable=~frame.get("sample_reliable",pd.Series(True,index=frame.index)).fillna(False).astype(bool)
    quality_rows.append({"machine_id":frame.machine_id.iloc[0],"run_id":run_id,"relationship_to_difficult":"difficult" if run_id==DIFFICULT_RUN else "preceding_development_neighbor",
        "rows":len(frame),"segments":frame.segment_id.nunique(),"invalid_timestamp_count":int(frame._timestamp_dt.isna().sum()),
        "duplicate_timestamp_count":int(frame.duplicated(["machine_id","run_id","timestamp"]).sum()),"median_interval_seconds":median_interval,
        "p95_interval_seconds":positive_delta.quantile(.95),"large_gap_count":int(delta.gt(gap_threshold).sum()),"maximum_gap_seconds":delta.max(),
        "raw_missing_percentage":float(frame[quality_metrics].isna().sum().sum()/max(1,frame[quality_metrics].size)*100),
        "constant_raw_feature_count":int(sum(frame[c].nunique(dropna=False)<=1 for c in quality_metrics)),"impossible_metric_value_count":impossible,
        "sensor_abnormal_row_count":int(sensor_abnormal.sum()),"sensor_abnormal_percentage":float(sensor_abnormal.mean()*100),
        "sample_unreliable_row_count":int(sample_unreliable.sum()),"sample_unreliable_percentage":float(sample_unreliable.mean()*100),
        "missed_deadline_count":int(pd.to_numeric(frame.get("missed_deadline",0),errors="coerce").fillna(0).astype(bool).sum())})
quality=pd.DataFrame(quality_rows)
dq=quality.loc[quality.run_id.eq(DIFFICULT_RUN)].iloc[0]
major_quality_issue = dq.impossible_metric_value_count>0 or dq.invalid_timestamp_count>0 or dq.duplicate_timestamp_count>0 or dq.raw_missing_percentage>10 or dq.sensor_abnormal_percentage>10 or dq.sample_unreliable_percentage>10
quality_classification = "B_likely_data_quality_issue" if major_quality_issue else "A_likely_valid_real_regime"
quality["data_quality_classification"] = np.where(quality.run_id.eq(DIFFICULT_RUN),quality_classification,"comparison_neighbor")
quality.to_csv(REPORT_DIR / "difficult_run_data_quality.csv",index=False)
display(quality)

,machine_id,run_id,relationship_to_difficult,rows,segments,invalid_timestamp_count,duplicate_timestamp_count,median_interval_seconds,p95_interval_seconds,large_gap_count,maximum_gap_seconds,raw_missing_percentage,constant_raw_feature_count,impossible_metric_value_count,sensor_abnormal_row_count,sensor_abnormal_percentage,sample_unreliable_row_count,sample_unreliable_percentage,missed_deadline_count,data_quality_classification
0,a0f8c86097e55fbfa506d057,c89dd147-c631-4848-a604-72c697e9da2b,preceding_development_neighbor,2118,2,0,0,4.094,8.1674,1,112.864,0.037771,1,0,10,0.472144,889,41.97356,889,comparison_neighbor
1,a0f8c86097e55fbfa506d057,0fc9db54-83d3-456a-b9ea-1aa8a6f66cf9,preceding_development_neighbor,38,1,0,0,8.997,12.3290,0,14.716,0.789474,2,0,38,100.000000,38,100.00000,38,comparison_neighbor
2,a0f8c86097e55fbfa506d057,baa2a6f4-7121-4bce-9c80-611ebc22ceaf,difficult,9188,8,0,0,2.045,4.2320,7,4521.461,0.168154,1,0,303,3.297780,739,8.04310,453,A_likely_valid_real_regime


### 8. Synthesize the diagnostic conclusion and practical path

**What this cell does:** Combines shift strength, target-conditional differences, probability behavior, Rule C components, machine variability, and quality evidence into a compact evidence table.  
**Why it matters:** The project needs one bounded practical recommendation rather than another open-ended modeling cycle.  
**What to understand:** The classification is a diagnostic judgment, not an automatic pipeline change, and the recommendation is deliberately not implemented.

In [8]:
top20_summary=(feature_ranking.groupby("comparison",group_keys=False).head(20).groupby("comparison").agg(
    median_top20_shift=("shift_metric","median"),mean_top20_shift=("shift_metric","mean"),median_top20_abs_smd=("absolute_smd","median"),median_top20_ks=("ks_statistic","median")
).reset_index())
shift_lookup=top20_summary.set_index("comparison")["median_top20_shift"]
general_shift=shift_lookup["train_vs_difficult"]
positive_shift=shift_lookup["train_positive_vs_difficult_positive"]
negative_shift=shift_lookup["train_negative_vs_difficult_negative"]
positive_probs=dp.loc[dp.true_target.eq(1),prob_col]
confident_fn_rate=float((positive_probs<.2).mean())
rule_now_frequency = rule_c_profile.query("profile_type=='signal_frequency' and condition=='rule_c_now_recreated'").set_index('group')['triggered_percentage']
rule_c_positive_alignment_gap = float(abs(rule_now_frequency['train_positive_future_label'] - rule_now_frequency['difficult_positive_future_label']))

signal_pos=rule_c_profile.query("profile_type=='signal_frequency' and group=='difficult_positive_future_label' and condition!='rule_c_now_recreated'").sort_values("triggered_percentage",ascending=False)
dominant_pressures=", ".join(signal_pos.head(3).condition.tolist())
machine_a0=machine_summary[machine_summary.machine_id.astype(str).str.startswith("a0f8")].iloc[0]
machine_0890=machine_summary[machine_summary.machine_id.astype(str).str.startswith("0890")].iloc[0]

if quality_classification.startswith("B_"):
    problem_class="C_mostly_data_quality_problem"; practical_path="PATH_3_correct_data_quality_before_continuing"
elif rule_c_positive_alignment_gap >= 20 and confident_fn_rate >= .20:
    problem_class="B_possible_concept_drift_with_strong_covariate_shift"; practical_path="PATH_4_keep_experimental_prototype_and_proceed_carefully_to_demo"
elif positive_shift > 1.25*negative_shift:
    problem_class="B_possible_concept_drift"; practical_path="PATH_2_consider_small_target_conditional_robustness_correction"
else:
    problem_class="A_mostly_covariate_shift"; practical_path="PATH_1_document_valid_shift_and_continue_cautiously_to_threshold_review"

summary_rows=[
    ("scope","feature_dataset_rows",feature_row_count,"Current 153k experiment; development-only diagnostics"),
    ("quality","difficult_run_classification",quality_classification,"Compared with preceding development neighbors; no automatic removal"),
    ("shift","problem_classification",problem_class,"Based on overall and target-conditional shift plus data quality"),
    ("shift","median_top20_overall_shift",general_shift,"Composite |SMD| + capped PSI + KS"),
    ("shift","median_top20_positive_shift",positive_shift,"Train positives versus difficult-run positives"),
    ("shift","median_top20_negative_shift",negative_shift,"Train negatives versus difficult-run negatives"),
    ("rule_c","dominant_difficult_positive_pressures",dominant_pressures,"Exact unchanged Rule C component frequencies"),
    ("rule_c","future_positive_current_rule_c_alignment_gap_percentage_points",rule_c_positive_alignment_gap,"Training-positive minus difficult-positive current pressure alignment"),
    ("probability","difficult_positive_below_0_2_rate",confident_fn_rate,"Confidently low probability among actual positives"),
    ("machine","machine_0890_run_positive_rate_range",f"{machine_0890.run_positive_rate_min:.4f}–{machine_0890.run_positive_rate_max:.4f}","Development runs only"),
    ("machine","machine_a0f8_run_positive_rate_range",f"{machine_a0.run_positive_rate_min:.4f}–{machine_a0.run_positive_rate_max:.4f}","Development runs only"),
    ("decision","practical_completion_path",practical_path,"Recommendation only; not implemented"),
    ("decision","another_model_cycle_required","not_for_project_completion_collect_more_regimes_before_production","No retraining or tuning performed"),
    ("guardrail","test_model_performance_used",False,"No test predictions or labels loaded"),
]
distribution_summary=pd.DataFrame(summary_rows,columns=["section","metric","value","interpretation"])
distribution_summary.to_csv(REPORT_DIR / "distribution_shift_summary.csv",index=False)
display(top20_summary)
display(distribution_summary)

,comparison,median_top20_shift,mean_top20_shift,median_top20_abs_smd,median_top20_ks
0,train_negative_vs_difficult_negative,7.387619,7.244755,1.609578,0.777688
1,train_positive_vs_difficult_positive,6.642893,6.514278,1.062823,0.570363
2,train_vs_difficult,7.043691,6.792467,1.336255,0.707267


,section,metric,value,interpretation
0,scope,feature_dataset_rows,133016,Current 153k experiment; development-only diag...
1,quality,difficult_run_classification,A_likely_valid_real_regime,Compared with preceding development neighbors;...
2,shift,problem_classification,B_possible_concept_drift_with_strong_covariate...,Based on overall and target-conditional shift ...
3,shift,median_top20_overall_shift,7.043691,Composite |SMD| + capped PSI + KS
4,shift,median_top20_positive_shift,6.642893,Train positives versus difficult-run positives
5,shift,median_top20_negative_shift,7.387619,Train negatives versus difficult-run negatives
6,rule_c,dominant_difficult_positive_pressures,"moderate_ram, moderate_context, moderate_cpu",Exact unchanged Rule C component frequencies
7,rule_c,future_positive_current_rule_c_alignment_gap_p...,36.469807,Training-positive minus difficult-positive cur...
8,probability,difficult_positive_below_0_2_rate,0.301088,Confidently low probability among actual posit...
9,machine,machine_0890_run_positive_rate_range,0.0000–1.0000,Development runs only


### 9. Visualize the most important distribution and drift patterns

**What this cell does:** Creates focused figures for feature shifts, raw metrics, target-positive regimes, model probabilities, run chronology, feature drift, and machine differences.  
**Why it matters:** Visual structure can reveal multimodality, recurring regimes, and boundary behavior that summary averages conceal.  
**What to understand:** Every plot is diagnostic and uses development data only; no threshold or model is selected from these figures.

In [9]:
plt.style.use("seaborn-v0_8-whitegrid")
top=feature_ranking.query("comparison=='train_vs_difficult'").head(20).sort_values("shift_metric")
plt.figure(figsize=(10,7)); plt.barh(top.feature,top.shift_metric,color="#4c78a8"); plt.xlabel("Composite shift score"); plt.title("Top shifted features: train vs difficult run"); plt.tight_layout(); plt.savefig(FIGURE_DIR/"top_shifted_features.png",dpi=160); plt.close()

plot_raw=[c for c in ["cpu_pct","ram_pct","swap_pct","disk_latency_ms","context_switches_per_s"] if c in RAW_FEATURES]
sample_train=train.sample(min(10000,len(train)),random_state=42).assign(group="train")
sample_diff=difficult.sample(min(8602,len(difficult)),random_state=42).assign(group="difficult run")
fig,axes=plt.subplots(len(plot_raw),1,figsize=(10,3*len(plot_raw)))
for ax,feature in zip(np.atleast_1d(axes),plot_raw):
    for frame,label,color in [(sample_train,"train","#4c78a8"),(sample_diff,"difficult run","#e45756")]:
        values=np.sort(pd.to_numeric(frame[feature],errors="coerce").dropna().to_numpy()); ax.plot(values,np.arange(1,len(values)+1)/len(values),label=label,color=color)
    ax.set_title(feature); ax.set_ylabel("ECDF"); ax.legend()
fig.tight_layout(); fig.savefig(FIGURE_DIR/"raw_feature_distributions.png",dpi=150); plt.close(fig)

pos_plot=pd.concat([groups["train_positive"].sample(min(8000,len(groups["train_positive"])),random_state=42).assign(group="train positive"), groups["difficult_positive"].assign(group="difficult positive")])
long=[]
for feature in plot_raw:
    center=train[feature].median(); scale=train[feature].quantile(.75)-train[feature].quantile(.25)
    temp=pos_plot[["group",feature]].copy(); temp["standardized_value"]=(temp[feature]-center)/(scale if scale else 1); temp["feature"]=feature; long.append(temp[["group","feature","standardized_value"]])
long=pd.concat(long); long["standardized_value"]=long.standardized_value.clip(-5,5)
fig,ax=plt.subplots(figsize=(11,6)); positions=[]; labels=[]
for i,feature in enumerate(plot_raw):
    for offset,group,color in [(-.18,"train positive","#4c78a8"),(.18,"difficult positive","#e45756")]:
        values=long[(long.feature==feature)&(long.group==group)].standardized_value.dropna(); bp=ax.boxplot(values,positions=[i+offset],widths=.30,showfliers=False,patch_artist=True); bp['boxes'][0].set_facecolor(color)
    labels.append(feature)
ax.set_xticks(range(len(plot_raw)),labels,rotation=25); ax.set_ylabel("train-IQR standardized value"); ax.set_title("Target-positive raw regimes"); ax.legend([plt.Rectangle((0,0),1,1,color="#4c78a8"),plt.Rectangle((0,0),1,1,color="#e45756")],["train positive","difficult positive"]); fig.tight_layout(); fig.savefig(FIGURE_DIR/"train_positive_vs_difficult_positive.png",dpi=160); plt.close(fig)

plt.figure(figsize=(9,5))
for outcome,color in [("true_positive","#4c78a8"),("false_negative","#e45756")]: plt.hist(dp.loc[dp.diagnostic_outcome.eq(outcome),prob_col],bins=30,density=True,histtype="step",linewidth=2,label=outcome,color=color)
plt.axvline(.5,color="black",ls="--"); plt.legend(); plt.title("Difficult-run positive probabilities: detected vs missed"); plt.tight_layout(); plt.savefig(FIGURE_DIR/"difficult_run_probability_tp_fn.png",dpi=160); plt.close()

machines=run_distribution_drift.machine_id.unique(); fig,axes=plt.subplots(len(machines),1,figsize=(12,3*len(machines)),sharex=False)
for ax,(machine,frame) in zip(np.atleast_1d(axes),run_distribution_drift.groupby("machine_id",sort=False)):
    ax.plot(frame.chronological_position,frame.positive_rate,marker="o"); ax.set_ylim(-.05,1.05); ax.set_title(machine); ax.set_ylabel("positive rate");
    hit=frame.run_id.eq(DIFFICULT_RUN); ax.scatter(frame.loc[hit,"chronological_position"],frame.loc[hit,"positive_rate"],s=140,color="red",label="difficult run");
    if hit.any(): ax.legend()
axes[-1].set_xlabel("Development run chronological position"); fig.tight_layout(); fig.savefig(FIGURE_DIR/"positive_rate_by_run_over_time.png",dpi=160); plt.close(fig)

drift_features=[c for c in ["cpu_pct_p90","ram_pct_p90","disk_latency_ms_p90","context_switches_per_s_p90"] if c in run_distribution_drift]
drift=run_distribution_drift[["machine_id","chronological_position"]+drift_features].copy()
for c in drift_features: drift[c]=(drift[c]-drift[c].median())/(drift[c].quantile(.75)-drift[c].quantile(.25) or 1)
fig,axes=plt.subplots(len(machines),1,figsize=(12,3*len(machines)))
for ax,(machine,frame) in zip(np.atleast_1d(axes),drift.groupby("machine_id",sort=False)):
    for c in drift_features: ax.plot(frame.chronological_position,frame[c],marker=".",label=c.replace("_p90",""))
    ax.set_title(machine); ax.set_ylabel("robust standardized p90"); ax.legend(ncol=4,fontsize=8)
axes[-1].set_xlabel("Development run chronological position"); fig.tight_layout(); fig.savefig(FIGURE_DIR/"feature_drift_by_run.png",dpi=160); plt.close(fig)

heat=machine_summary.set_index("machine_id")[[f"{c}_median" for c in major_raw]]
heat=(heat-heat.mean())/heat.std(ddof=0).replace(0,1)
fig,ax=plt.subplots(figsize=(12,4)); image=ax.imshow(heat.to_numpy(),aspect="auto",cmap="coolwarm",vmin=-np.nanmax(abs(heat.to_numpy())),vmax=np.nanmax(abs(heat.to_numpy()))); ax.set_xticks(range(len(heat.columns)),[c.replace("_median","") for c in heat.columns],rotation=35,ha="right"); ax.set_yticks(range(len(heat.index)),heat.index);
for i in range(len(heat.index)):
    for j in range(len(heat.columns)): ax.text(j,i,f"{heat.iloc[i,j]:.1f}",ha="center",va="center",fontsize=8)
fig.colorbar(image,ax=ax,label="column-standardized median"); ax.set_title("Machine raw-feature medians"); fig.tight_layout(); fig.savefig(FIGURE_DIR/"machine_level_comparison.png",dpi=160); plt.close(fig)
print(f"Saved {len(list(FIGURE_DIR.glob('*.png')))} figures to {FIGURE_DIR}")

Saved 7 figures to /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/figures/distribution_shift


### 10. Verify immutability and print the bounded final diagnosis

**What this cell does:** Rechecks all input and test checksums, validates every required report, and prints the key conclusion and stop condition.  
**Why it matters:** A diagnostic notebook must prove that it did not mutate data or cross into test evaluation, retraining, or threshold work.  
**What to understand:** Successful assertions mean the analysis is reproducible and the existing pipeline remains unchanged.

In [10]:
required_outputs = [
    "distribution_shift_summary.csv","distribution_shift_feature_ranking.csv","difficult_run_rule_c_profile.csv",
    "difficult_run_probability_profile.csv","difficult_run_false_negative_profile.csv","machine_distribution_summary.csv",
    "run_distribution_drift.csv","difficult_run_data_quality.csv"
]
input_checksums_after={name:sha256(path) for name,path in PATHS.items()}
test_checksums_after={str(path.relative_to(PROJECT_ROOT)):sha256(path) for path in TEST_PATHS}
assert input_checksums_before == input_checksums_after, "An input artifact changed during diagnosis."
assert test_checksums_before == test_checksums_after, "A protected test artifact changed."
for filename in required_outputs:
    path=REPORT_DIR/filename
    assert path.exists() and path.stat().st_size>0
    assert len(pd.read_csv(path))>0

top_overall=feature_ranking.query("comparison=='train_vs_difficult'").head(5).feature.tolist()
below_02=int(probability_profile.loc[probability_profile.group.eq("true_positive_actual"),"count_probability_below_0_2"].iloc[0])
actual_pos=int((dp.true_target==1).sum())
print("FINAL DISTRIBUTION-SHIFT DIAGNOSIS")
print(f"Difficult run quality: {quality_classification}")
print(f"Primary diagnosis: {problem_class}")
print(f"Top overall shifted features: {', '.join(top_overall)}")
print(f"Actual difficult-run positives below probability 0.20: {below_02:,}/{actual_pos:,} ({below_02/actual_pos:.2%})")
print(f"Dominant Rule C components among difficult-run future positives: {dominant_pressures}")
print(f"Recommended practical path: {practical_path}")
print("Inputs unchanged: True; protected TEST unchanged byte-for-byte: True")
print("STOP: no retraining, tuning, threshold optimization, calibration, test evaluation, SHAP, or dashboard work was performed.")

FINAL DISTRIBUTION-SHIFT DIAGNOSIS
Difficult run quality: A_likely_valid_real_regime
Primary diagnosis: B_possible_concept_drift_with_strong_covariate_shift
Top overall shifted features: disk_free_gb, disk_usage_pct, thread_count_min_120s, thread_count_mean_120s, thread_count_min_60s
Actual difficult-run positives below probability 0.20: 526/1,747 (30.11%)
Dominant Rule C components among difficult-run future positives: moderate_ram, moderate_context, moderate_cpu
Recommended practical path: PATH_4_keep_experimental_prototype_and_proceed_carefully_to_demo
Inputs unchanged: True; protected TEST unchanged byte-for-byte: True
STOP: no retraining, tuning, threshold optimization, calibration, test evaluation, SHAP, or dashboard work was performed.
